In [1]:
#!/usr/bin/env python

import os
import shutil
import queue
import threading
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import zarr
import tqdm
from torch.utils.data import Dataset, DataLoader
import threading


In [2]:

# ==================================================================
# ===== 1. MODELL- UND HELFER-DEFINITIONEN (aus Ihrem Training) =====
# ==================================================================

### IHRE ORIGINALE UNet3D ARCHITEKTUR ###
class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)
        
        # Initialize final layer to predict zero displacement
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.upconv2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final_conv(d1)

### STANDARD-IMPLEMENTIERUNG DES SPATIAL TRANSFORMERS ###
class SpatialTransformer3D(nn.Module):
    def __init__(self, size):
        super(SpatialTransformer3D, self).__init__()
        self.size = size # Erwartete Größe: (D, H, W)
        vectors = [torch.arange(0, s) for s in size]
        grids = torch.meshgrid(vectors, indexing='ij')
        grid = torch.stack(grids)
        grid = torch.unsqueeze(grid, 0)
        grid = grid.type(torch.FloatTensor)
        self.register_buffer('grid', grid)

    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]
        for i in range(len(shape)):
            new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)
        new_locs = new_locs.permute(0, 2, 3, 4, 1)
        new_locs = new_locs[..., [2, 1, 0]]
        return F.grid_sample(src, new_locs, align_corners=True, mode='bilinear')

### ASYNCHRONER WRITER-THREAD ###
def save_to_zarr_worker(q, warped_zarr, dvf_zarr):
    while True:
        data = q.get()
        if data is None: break
        time_idx, warped_np, dvf_np = data
        try:
            warped_zarr[..., time_idx] = warped_np
            dvf_zarr[..., time_idx, :] = dvf_np
        except Exception as e:
            print(f"Fehler im Writer-Thread beim Index {time_idx}: {e}")
        q.task_done()


In [3]:
class SelfRegistrationDataset(Dataset):
    """
    KORRIGIERTE VERSION: Liest aus einem "flachen" 3D Zarr-Array (Frames, H, W)
    und rekonstruiert 3D-Volumen (D, H, W) on-the-fly.
    """
    def __init__(self, moving_path, fixed_index=0, num_slices_per_volume=50, k=4):
        print(f"Lade 'flaches' 3D Zarr-Array von: {moving_path}")
        self.moving_arr = zarr.open(moving_path, mode='r')
        self.fixed_index = fixed_index
        self.D = num_slices_per_volume # Anzahl der Schichten pro Volumen
        self.k = k

        assert self.moving_arr.ndim == 3, f"Erwartet wird ein 3D Array (Frames, H, W), erhalten wurde {self.moving_arr.ndim}D"
        
        self.num_time_points = self.moving_arr.shape[0] // self.D
        self.original_shape = (self.moving_arr.shape[1], self.moving_arr.shape[2], self.D) # (H, W, D)
        
        self.padded_shape = self._calculate_padded_shape(self.original_shape)

        print(f"Originale 3D-Form (H, W, D): {self.original_shape}")
        print(f"Gepaddete 3D-Form: {self.padded_shape}")

        print(f"Lade und bereite festes Referenzbild (Zeitschritt {self.fixed_index}) vor...")
        fixed_volume_np = self._get_volume(self.fixed_index) # (D, H, W)
        fixed_tensor_unpadded = torch.from_numpy(fixed_volume_np.astype(np.float32)).unsqueeze(0) # (1, D, H, W)
        self.fixed_tensor = self._pad_tensor(fixed_tensor_unpadded)

    def _get_volume(self, time_idx):
        start_slice = time_idx * self.D
        end_slice = start_slice + self.D
        # Lädt alle Schichten für einen Zeitpunkt und hat die Form (D, H, W)
        volume_np = self.moving_arr[start_slice:end_slice, :, :] 
        return volume_np

    def _calculate_padded_shape(self, shape): # (H, W, D)
        return tuple([(s + self.k - 1) // self.k * self.k for s in shape])

    def _pad_tensor(self, tensor): # (C, D, H, W)
        shape_in = tensor.shape
        shape_out = (1, self.padded_shape[2], self.padded_shape[0], self.padded_shape[1]) # (C, D_pad, H_pad, W_pad)
        pad_d = shape_out[1] - shape_in[1]
        pad_h = shape_out[2] - shape_in[2]
        pad_w = shape_out[3] - shape_in[3]
        padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_d // 2, pad_d - pad_d // 2)
        return F.pad(tensor, padding, "constant", 0)

    def __len__(self):
        return self.num_time_points

    def __getitem__(self, idx):
        moving_volume_np = self._get_volume(idx) # (D, H, W)
        moving_tensor_unpadded = torch.from_numpy(moving_volume_np.astype(np.float32)).unsqueeze(0) # (1, D, H, W)
        moving_tensor = self._pad_tensor(moving_tensor_unpadded)
        return moving_tensor, self.fixed_tensor

In [4]:
DATASET_NAME = '159269_B1'
PART = '14'

In [5]:

# Define paths (please adjust if needed)
dicom_folder = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/DICOM' 
zarr_file = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/{DATASET_NAME}_{PART}.zarr'
file_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/AIF_2.txt'
results_mococo_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/{DATASET_NAME}_{PART}_warped/'
results_warped_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/{DATASET_NAME}_{PART}_dvs/'

In [6]:

# ==================================================================
# ======================= 3. HAUPTSKRIPT =========================
# ==================================================================
if __name__ == '__main__':
    # --- Konfiguration ---
    MODEL_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/best_supervised_model_250810_02_batch8_LR1e-5_Ep100_VSpl1_RegWeight01.pth"
    # Annahme: zarr_file, results_warped_path und results_mococo_path sind bereits definiert
    zarr_file = zarr_file # BITTE ANPASSEN
    results_warped_path = results_warped_path # BITTE ANPASSEN
    results_mococo_path = results_mococo_path # BITTE ANPASSEN

    MOVING_TEST_PATH = zarr_file
    OUTPUT_WARPED_PATH = results_warped_path
    OUTPUT_DVF_PATH = results_mococo_path
    FIXED_IMAGE_INDEX = 1
    NUM_WRITER_THREADS = 8

    # --- Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Verwende Gerät: {device}")
    
    for path in [OUTPUT_WARPED_PATH, OUTPUT_DVF_PATH]:
        if os.path.exists(path):
            if os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
            print(f"Warnung: Bestehende Datei/Verzeichnis {path} wurde gelöscht.")

    # --- Daten und Modell laden ---
    print("\nLade Datensatz...")
    dataset = SelfRegistrationDataset(
        MOVING_TEST_PATH, 
        fixed_index=FIXED_IMAGE_INDEX,
        num_slices_per_volume=50  # Hier die Anzahl der Schichten pro 3D-Volumen angeben
    )
    
    padded_input_size = (dataset.padded_shape[2], dataset.padded_shape[0], dataset.padded_shape[1]) # (D, H, W)
    
    print("\nLade trainiertes Modell...")
    print("\nLade trainiertes Modell...")
    model = UNet3D().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device)) # Sollte jetzt funktionieren
    print("Modell-Gewichte erfolgreich geladen.")

    try:
        model = torch.compile(model)
        print("Modell erfolgreich mit torch.compile() optimiert.")
    except Exception: 
        print("torch.compile() nicht verfügbar oder fehlgeschlagen.") # DIESER ZWEIG WIRD AUSGEFÜHRT
    
    stn = SpatialTransformer3D(size=padded_input_size).to(device)
    
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=max(1, os.cpu_count() // 2), pin_memory=True, prefetch_factor=2)

    # --- Output-Dateien und Writer-Threads ---
    print("\nErstelle thread-sichere Ausgabedateien...")

    synchronizer = threading.Lock()

    warped_output_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='w', 
                                   shape=dataset.moving_arr.shape, 
                                   chunks=dataset.moving_arr.chunks, 
                                   dtype=dataset.moving_arr.dtype, 
                                   compressor=None, 
                                   synchronizer=synchronizer,
                                   zarr_version=2) # <--- KORREKTUR

    dvf_shape = dataset.moving_arr.shape + (3,)
    dvf_chunks = dataset.moving_arr.chunks + (3,)

    dvf_output_zarr = zarr.open(OUTPUT_DVF_PATH, mode='w', 
                                   shape=dvf_shape, 
                                   chunks=dvf_chunks, 
                                   dtype='float32', 
                                   compressor=None, 
                                   synchronizer=synchronizer,
                                   zarr_version=2) # <--- KORREKTUR
    
    data_queue = queue.Queue(maxsize=NUM_WRITER_THREADS * 8)
    writer_threads = []
    print(f"Starte {NUM_WRITER_THREADS} asynchrone Writer-Threads...")
    for _ in range(NUM_WRITER_THREADS):
        thread = threading.Thread(target=save_to_zarr_worker, args=(data_queue, warped_output_zarr, dvf_output_zarr))
        thread.daemon = True
        thread.start(); writer_threads.append(thread)

    # --- Inferenz-Schleife ---
    print(f"\nStarte optimierte Inferenz für {len(dataset)} Volumen...")
    original_shape_dims = dataset.original_shape # (H, W, D)
    padded_shape_dims = dataset.padded_shape   # (H_pad, W_pad, D_pad)
    model.eval()
    with torch.no_grad():
        for i, (moving_batch, fixed_batch) in enumerate(tqdm.tqdm(dataloader, desc="Verarbeite Testdatensatz")):
            moving_batch = moving_batch.to(device, non_blocking=True); fixed_batch = fixed_batch.to(device, non_blocking=True)
            
            with torch.autocast(device_type=str(device), dtype=torch.float16):
                predicted_dvf_batch = model(fixed_batch, moving_batch)
                warped_batch = stn(moving_batch, predicted_dvf_batch)
            
            warped_tensor = warped_batch.squeeze(0).cpu(); disp_field = predicted_dvf_batch.squeeze(0).cpu()
            
            pad_h = padded_shape_dims[0] - original_shape_dims[0]
            pad_w = padded_shape_dims[1] - original_shape_dims[1]
            pad_d = padded_shape_dims[2] - original_shape_dims[2]
            
            pad_h_start = pad_h // 2
            pad_w_start = pad_w // 2
            pad_d_start = pad_d // 2

            cropped_warped = warped_tensor[pad_d_start:pad_d_start+original_shape_dims[2], 
                                           pad_h_start:pad_h_start+original_shape_dims[0], 
                                           pad_w_start:pad_w_start+original_shape_dims[1]]
            
            cropped_disp = disp_field[:, 
                                      pad_d_start:pad_d_start+original_shape_dims[2], 
                                      pad_h_start:pad_h_start+original_shape_dims[0], 
                                      pad_w_start:pad_w_start+original_shape_dims[1]]
            
            warped_np = cropped_warped.numpy().transpose(1, 2, 0)
            dvf_np = cropped_disp.numpy().transpose(2, 3, 1, 0)
            
            data_queue.put((i, warped_np, dvf_np))
            
    # --- Aufräumen ---
    print("\nHauptprozess abgeschlossen. Sende Stopp-Signal an Writer-Threads...")
    for _ in range(NUM_WRITER_THREADS): data_queue.put(None)
    
    print("Warte, bis alle Writer-Threads ihre Arbeit beendet haben...")
    for thread in writer_threads: thread.join()

    print(f"\nVerarbeitung und Speicherung in {OUTPUT_WARPED_PATH} und {OUTPUT_DVF_PATH} abgeschlossen.")

Verwende Gerät: cpu
Warnung: Bestehende Datei/Verzeichnis /mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1_14_dvs/ wurde gelöscht.
Warnung: Bestehende Datei/Verzeichnis /mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/159269_B1_14_warped/ wurde gelöscht.

Lade Datensatz...
Lade 'flaches' 3D Zarr-Array von: /mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/159269_B1/14/159269_B1_14.zarr
Originale 3D-Form (H, W, D): (256, 256, 50)
Gepaddete 3D-Form: (256, 256, 52)
Lade und bereite festes Referenzbild (Zeitschritt 1) vor...

Lade trainiertes Modell...

Lade trainiertes Modell...
Modell-Gewichte erfolgreich geladen.


/home/shooty/anaconda3/envs/ds-env-01/lib/python3.13/site-packages/zarr/api/asynchronous.py:1267: RuntimeWarning: synchronizer is not yet implemented
  return await create(


Modell erfolgreich mit torch.compile() optimiert.

Erstelle thread-sichere Ausgabedateien...
Starte 8 asynchrone Writer-Threads...

Starte optimierte Inferenz für 250 Volumen...


Verarbeite Testdatensatz:   0%|          | 0/250 [00:00<?, ?it/s]/home/shooty/anaconda3/envs/ds-env-01/lib/python3.13/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Verarbeite Testdatensatz:   0%|          | 0/250 [00:11<?, ?it/s]


InductorError: ImportError: /tmp/torchinductor_shooty/ez/cezalkghyo3mctsgv4p6muipz4cbwibckomjn325d5cpiebal7gs.so: undefined symbol: __cxa_call_terminate

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"
